In [ ]:
import numpy as np

# Step 1: Raw readings — 12 observations (rows) x 4 sensors (columns)
# Two implausible values are deliberately included:
#   - row index 5, sensor B: a spike (990 raw counts)
#   - row index 11, sensor D: a dropout (15 raw counts)
sensor_readings = np.array([
    [512, 208, 715, 1005],
    [518, 203, 728, 1012],
    [505, 210, 705,  998],
    [522, 199, 733, 1020],
    [509, 206, 718, 1003],
    [515, 990, 722, 1008],   # sensor B spike
    [520, 201, 710,  995],
    [507, 207, 726, 1015],
    [513, 204, 719, 1006],
    [519, 202, 731, 1018],
    [504, 209, 708,  999],
    [516, 205, 720,   15],   # sensor D dropout
], dtype=float)

print("Shape (observations, sensors):", sensor_readings.shape)

# Step 2: Apply a per-sensor scale and offset via broadcasting.
# sensor_scales/sensor_offsets have shape (4,), which aligns with the
# trailing (column) axis of the (12, 4) array — one factor per sensor.
sensor_scales  = np.array([0.01, 0.05, 0.02, 0.005])
sensor_offsets = np.array([-0.20, 1.00, -5.00, -0.50])

calibrated = sensor_readings * sensor_scales + sensor_offsets
print("Calibrated shape:", calibrated.shape)
print(np.round(calibrated, 3))

# Step 3: Flag implausible values with a Boolean mask.
# Plausible range documented per sensor (engineering units after calibration):
#   Sensor A: 2.5 - 5.5 V     Sensor B: 2.5 - 12.0 %RH
#   Sensor C: 8.0 - 17.0 deg C   Sensor D: 3.0 - 6.0 bar
lower_bounds = np.array([2.5, 2.5, 8.0, 3.0])
upper_bounds = np.array([5.5, 12.0, 17.0, 6.0])

valid_mask = (calibrated >= lower_bounds) & (calibrated <= upper_bounds)
print("Valid mask:")
print(valid_mask)

# Step 4: Per-sensor summaries.
# axis=0 collapses the 12 observations and keeps the 4 sensor columns,
# so each result below is "one value per sensor" (length-4 arrays).
valid_counts    = valid_mask.sum(axis=0)                     # per-sensor valid count
rejected_counts = (~valid_mask).sum(axis=0)                  # per-sensor rejected count
masked_values   = np.where(valid_mask, calibrated, np.nan)   # invalid entries become NaN
mean_per_sensor = np.nanmean(masked_values, axis=0)          # mean ignoring NaN, per sensor
std_per_sensor  = np.nanstd(masked_values, axis=0)           # spread ignoring NaN, per sensor

assert np.array_equal(valid_counts + rejected_counts, np.full(4, sensor_readings.shape[0]))

report = {
    "sensor_labels": ["A", "B", "C", "D"],
    "valid_count_per_sensor": valid_counts.tolist(),
    "rejected_count_per_sensor": rejected_counts.tolist(),
    "mean_valid_per_sensor": np.round(mean_per_sensor, 3).tolist(),
    "std_valid_per_sensor": np.round(std_per_sensor, 3).tolist(),
}
report